# European Trade Network Dataset

*Natalie Montanez, Yirang Liu, Jason Wang*  
*April 14, 2026*

## Pipeline
```
Step 0   Setup
Step 1   Country scope (World Bank) → countries.csv
Step 2   Geocoding & metadata → countries.csv
Step 3   Node table (World Bank indicators) → nodes.csv
Step 4a  API pull → eu_comtrade.db
Step 4b  SQL aggregation → bilateral_edges.db → edges.csv
Step 5   Merge → european_trade.db + SCHEMA.md
Step 6   Validation & example queries
Step 7   Visualization
```

## Step 0. Setup

In [ ]:
import os, sqlite3, time, csv, json, random, warnings, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import numpy as np
import pandas as pd
import pycountry
import requests
import wbgapi as wb
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

COUNTRIES_CSV = DATA_DIR / "countries.csv"
NODES_CSV     = DATA_DIR / "nodes.csv"
EDGES_CSV     = DATA_DIR / "edges.csv"
DB_RAW        = DATA_DIR / "eu_comtrade.db"
DB_EDGES      = DATA_DIR / "bilateral_edges.db"
DB_FINAL      = DATA_DIR / "european_trade.db"
SCHEMA_PATH   = DATA_DIR / "SCHEMA.md"
FIG_PATH      = DATA_DIR / "network_2023.png"

YEARS = list(range(2019, 2024))
YEAR_FOR_VIS = 2023
print("Outputs:", DATA_DIR.resolve())

## Step 1. Country scope

World Bank ECS region minus Central Asia.

In [ ]:
CENTRAL_ASIA_ISO3 = {"KAZ", "UZB", "TKM", "TJK", "KGZ", "ARM", "AZE", "GEO"}


def build_countries_base():
    rows = []
    for iso3 in list(wb.region.members("ECS")):
        try:
            info = wb.economy.get(iso3)
            rows.append({"iso3": iso3, "countryname": info["value"]})
        except Exception as exc:
            print(f"Skipping {iso3}: {exc}")
    df = pd.DataFrame(rows)
    df = df[~df["iso3"].isin(CENTRAL_ASIA_ISO3)]
    df = df.sort_values("iso3").reset_index(drop=True)
    return df


countries = build_countries_base()
print(f"Countries: {len(countries)}")
countries.head()

## Step 2. Geocoding & metadata

Capital geocoding + manual patches + Comtrade codes.

In [ ]:
MANUAL_CAPITAL_PATCH = {
    "CHI": {"capital": "Saint Helier", "lat": 49.188, "lon": -2.101},
    "EST": {"capital": "Tallinn", "lat": 59.4370, "lon": 24.7536},
    "FRO": {"capital": "Torshavn", "lat": 62.0107, "lon": -6.7741},
    "GRL": {"capital": "Nuuk", "lat": 64.1835, "lon": -51.7216},
    "IMN": {"capital": "Douglas", "lat": 54.1524, "lon": -4.4861},
    "ISL": {"capital": "Reykjavik", "lat": 64.1355, "lon": -21.8954},
    "LIE": {"capital": "Vaduz", "lat": 47.1415, "lon": 9.5215},
    "MNE": {"capital": "Podgorica", "lat": 42.4304, "lon": 19.2594},
    "SMR": {"capital": "City of San Marino", "lat": 43.9361, "lon": 12.4463},
    "TUR": {"capital": "Ankara", "lat": 39.9334, "lon": 32.8597},
    "XKX": {"capital": "Pristina", "lat": 42.6629, "lon": 21.1655},
}

COORDINATE_OVERRIDES = {
    "ITA": {"capital": "Rome", "lat": 41.9028, "lon": 12.4964},
    "DEU": {"capital": "Berlin", "lat": 52.5200, "lon": 13.4050},
    "IRL": {"capital": "Dublin", "lat": 53.3498, "lon": -6.2603},
    "SWE": {"capital": "Stockholm", "lat": 59.3293, "lon": 18.0686},
    "HUN": {"capital": "Budapest", "lat": 47.4979, "lon": 19.0402},
    "ESP": {"capital": "Madrid", "lat": 40.4168, "lon": -3.7038},
    "CZE": {"capital": "Prague", "lat": 50.0755, "lon": 14.4378},
    "MDA": {"capital": "Chisinau", "lat": 47.0105, "lon": 28.8638},
    "LUX": {"capital": "Luxembourg", "lat": 49.6116, "lon": 6.1319},
    "NLD": {"capital": "Amsterdam", "lat": 52.3676, "lon": 4.9041},
}

COMTRADE_CODE_PATCH = {
    "CHI": 831,
    "XKX": 926,
}


def fetch_rest_capitals():
    url = "https://restcountries.com/v3.1/region/europe?fields=cca3,capital"
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    capital_map = {}
    for item in response.json():
        iso3 = item.get("cca3")
        capital_list = item.get("capital", [])
        if iso3 and capital_list:
            capital_map[iso3] = capital_list[0]
    return capital_map


def geocode_capitals(countries):
    capital_map = fetch_rest_capitals()
    geolocator = Nominatim(user_agent="europe_trade_network_final")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.2)

    rows = []
    for _, row in countries.iterrows():
        iso3 = row["iso3"]
        countryname = row["countryname"]
        capital = capital_map.get(iso3)
        lat = None
        lon = None

        if capital:
            try:
                location = geocode(f"{capital}, {countryname}")
                if location:
                    lat = location.latitude
                    lon = location.longitude
            except Exception:
                pass

        rows.append({"iso3": iso3, "capital": capital, "lat": lat, "lon": lon})

    geocoded = pd.DataFrame(rows)
    countries = countries.merge(geocoded, on="iso3", how="left")

    for iso3, patch in MANUAL_CAPITAL_PATCH.items():
        mask = countries["iso3"] == iso3
        countries.loc[mask, ["capital", "lat", "lon"]] = [patch["capital"], patch["lat"], patch["lon"]]

    for iso3, patch in COORDINATE_OVERRIDES.items():
        mask = countries["iso3"] == iso3
        countries.loc[mask, ["capital", "lat", "lon"]] = [patch["capital"], patch["lat"], patch["lon"]]

    countries["capital"] = countries["capital"].replace({"Chișinău": "Chisinau", "Chiinu": "Chisinau", "Tórshavn": "Torshavn", "Trshavn": "Torshavn"})
    return countries


def add_comtrade_codes(countries):
    rows = []
    for _, row in countries.iterrows():
        iso3 = row["iso3"]
        code = None
        try:
            country = pycountry.countries.get(alpha_3=iso3)
            if country and hasattr(country, "numeric"):
                code = int(country.numeric)
        except Exception:
            code = None
        rows.append({"iso3": iso3, "comtradenum": code})

    code_map = pd.DataFrame(rows)
    countries = countries.merge(code_map, on="iso3", how="left")
    countries["comtradenum"] = countries["comtradenum"].astype("Int64")

    for iso3, code in COMTRADE_CODE_PATCH.items():
        countries.loc[countries["iso3"] == iso3, "comtradenum"] = code

    countries["comtradenum"] = countries["comtradenum"].astype("Int64")
    return countries


countries = geocode_capitals(countries)
countries = add_comtrade_codes(countries)

countries = countries[["iso3", "countryname", "capital", "lat", "lon", "comtradenum"]].copy()
countries = countries.sort_values("iso3").reset_index(drop=True)
countries.to_csv(COUNTRIES_CSV, index=False)

print(f"Saved {COUNTRIES_CSV.name} with {len(countries)} rows")
print(countries[countries[["capital", "lat", "lon", "comtradenum"]].isna().any(axis=1)])
countries.head()


## Step 3. Node table (World Bank indicators)

9 small territories excluded (no World Bank data).

In [ ]:
WB_INDICATORS = {
    "NY.GDP.MKTP.CD": "gdpusd",
    "SP.POP.TOTL": "population",
    "NE.TRD.GNFS.ZS": "tradepctgdp",
    "BX.KLT.DINV.WD.GD.ZS": "fdiinflowpctgdp",
}

DROP_ISO3 = {"AND", "CHI", "FRO", "GIB", "GRL", "IMN", "LIE", "MCO", "SMR"}


def build_nodes(countries):
    iso3_list = countries[~countries["iso3"].isin(DROP_ISO3)]["iso3"].tolist()
    frames = []

    for indicator_code, column_name in WB_INDICATORS.items():
        df = wb.data.DataFrame(indicator_code, economy=iso3_list, time=YEARS, skipBlanks=False)
        df = df.reset_index().melt(id_vars="economy", var_name="year", value_name=column_name)
        df = df.rename(columns={"economy": "iso3"})
        df["year"] = df["year"].astype(str).str.extract(r"(\d{4})").astype(int)
        frames.append(df)

    nodes = frames[0]
    for df in frames[1:]:
        nodes = nodes.merge(df, on=["iso3", "year"], how="outer")

    nodes = nodes[nodes["year"].between(2019, 2023)].copy()
    nodes = nodes.dropna(subset=list(WB_INDICATORS.values())).copy()
    nodes = nodes.sort_values(["iso3", "year"]).reset_index(drop=True)
    return nodes


nodes = build_nodes(countries)
nodes.to_csv(NODES_CSV, index=False)

print(f"Saved {NODES_CSV.name} with {len(nodes)} rows")
print("Years covered:", sorted(nodes["year"].unique().tolist()))
nodes.head()


### Node validation

In [ ]:
def summarize_data_quality(nodes):
    print("Missing values by column:")
    print(nodes.isna().sum())
    print("Duplicate primary keys:", nodes.duplicated(subset=["iso3", "year"]).sum())
    print("Negative GDP rows:", (nodes["gdpusd"] < 0).sum())
    print("Negative population rows:", (nodes["population"] < 0).sum())
    print("Trade percent above 300:", (nodes["tradepctgdp"] > 300).sum())

summarize_data_quality(nodes)


## Step 4a. Pull API → eu_comtrade.db

50 countries × 5 years = 250 API calls (batch mode: one call per reporter, all partners at once).

⚠️ **Only run this cell once.** After first run, eu_comtrade.db is saved and you can skip to Step 4b.

In [ ]:
API_KEY = "ff3a4b4ef6b74058ab6866d2f6cd79cf"  # Replace with your key
BASE = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

# Build lookup tables from countries.csv
countries_ref = pd.read_csv(COUNTRIES_CSV)
code_to_info = {}
for _, row in countries_ref.iterrows():
    code_to_info[str(row["comtradenum"])] = {
        "name": row["countryname"],
        "iso3": row["iso3"]
    }
all_partner_codes = ",".join(
    str(c) for c in countries_ref["comtradenum"].dropna().astype(int).tolist()
)

# Create raw database
conn = sqlite3.connect(str(DB_RAW))
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS trade")
cur.execute("""
CREATE TABLE trade (
    reporter_iso3  TEXT,
    reporter_code  INT,
    reporter_name  TEXT,
    partner_iso3   TEXT,
    partner_code   INT,
    partner_name   TEXT,
    period         TEXT,
    flow_code      TEXT,
    cmd_code       TEXT,
    primary_value  REAL,
    cif_value      REAL,
    fob_value      REAL,
    net_wgt        REAL,
    partner2_code  INT,
    classification TEXT,
    is_reported    BOOLEAN,
    PRIMARY KEY (reporter_iso3, partner_iso3, period,
                 flow_code, cmd_code, partner2_code)
)
""")
cur.execute("CREATE INDEX idx_period ON trade(period)")
cur.execute("CREATE INDEX idx_flow ON trade(flow_code)")
cur.execute("CREATE INDEX idx_reporter_iso ON trade(reporter_iso3)")
cur.execute("CREATE INDEX idx_partner_iso ON trade(partner_iso3)")
conn.commit()


def fetch_and_store(reporter_code, reporter_name, reporter_iso3, year):
    params = {
        "reporterCode": str(reporter_code),
        "period": str(year),
        "partnerCode": all_partner_codes,
        "cmdCode": "AG2",
        "flowCode": "M,X",
        "subscription-key": API_KEY,
    }
    query = "&".join(f"{k}={v}" for k, v in params.items())
    url = f"{BASE}?{query}"
    req = urllib.request.Request(url, headers={"Cache-Control": "no-cache"})

    try:
        with urllib.request.urlopen(req) as resp:
            obj = json.loads(resp.read().decode("utf-8"))
    except Exception as e:
        print(f"  ERROR {reporter_name} ({year}): {e}")
        return 0

    count = 0
    for r in obj.get("data", []):
        pc = str(r["partnerCode"])
        p_info = code_to_info.get(pc, {"name": pc, "iso3": "UNK"})
        cur.execute(
            "INSERT OR REPLACE INTO trade VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (
                reporter_iso3, int(reporter_code), reporter_name,
                p_info["iso3"], r["partnerCode"], p_info["name"],
                r["period"], r["flowCode"], r["cmdCode"],
                r["primaryValue"], r.get("cifvalue"), r.get("fobvalue"),
                r.get("netWgt"), r.get("partner2Code", 0),
                r.get("classificationCode"), r.get("isReported"),
            ),
        )
        count += 1
    conn.commit()
    return count


# Run API calls
total_calls = 0
total_expected = len(countries_ref) * len(YEARS)

for year in YEARS:
    for _, row in countries_ref.iterrows():
        rc = str(int(row["comtradenum"]))
        n = fetch_and_store(rc, row["countryname"], row["iso3"], year)
        total_calls += 1
        print(f"[{total_calls}/{total_expected}] {row['iso3']} ({year}): {n} rows")
        time.sleep(1.2)

print(f"\nDone! {total_calls} calls. Saved {DB_RAW}")
conn.close()

## Step 4b. eu_comtrade.db → bilateral_edges.db → edges.csv

SQL aggregation: GROUP BY country pair × year × flow direction.

In [ ]:
conn = sqlite3.connect(str(DB_RAW))

# Raw data overview
overview = pd.read_sql("""
    SELECT period, flow_code, COUNT(*) AS rows,
           SUM(CASE WHEN primary_value IS NULL OR primary_value = 0
               THEN 1 ELSE 0 END) AS missing
    FROM trade
    GROUP BY period, flow_code
    ORDER BY period, flow_code
""", conn)
print("=== Raw data overview ===")
print(overview.to_string(index=False))

# Aggregate
edges = pd.read_sql("""
    SELECT reporter_iso3, reporter_name,
           partner_iso3, partner_name,
           period, flow_code,
           SUM(primary_value) AS total_trade_usd
    FROM trade
    GROUP BY reporter_iso3, partner_iso3, period, flow_code
""", conn)
conn.close()

print(f"\nAggregated edges: {len(edges)} rows")

# Save bilateral_edges.db
export_conn = sqlite3.connect(str(DB_EDGES))
edges.to_sql("edges", export_conn, if_exists="replace", index=False)
export_conn.close()

# Save edges.csv
edges.to_csv(EDGES_CSV, index=False)
print(f"Saved {EDGES_CSV.name} and {DB_EDGES.name}")
edges.head()

## Step 5. Merge → european_trade.db + SCHEMA.md

In [ ]:
countries_final = pd.read_csv(COUNTRIES_CSV)
nodes_final = pd.read_csv(NODES_CSV)
edges_final = pd.read_csv(EDGES_CSV)

print(f"countries: {len(countries_final)} rows")
print(f"nodes:     {len(nodes_final)} rows")
print(f"edges:     {len(edges_final)} rows")

# Write to database
conn = sqlite3.connect(str(DB_FINAL))
countries_final.to_sql("countries", conn, if_exists="replace", index=False)
nodes_final.to_sql("nodes", conn, if_exists="replace", index=False)
edges_final.to_sql("edges", conn, if_exists="replace", index=False)

# Indexes
cur = conn.cursor()
cur.execute("CREATE UNIQUE INDEX idx_countries_iso3 ON countries(iso3)")
cur.execute("CREATE INDEX idx_nodes_iso3 ON nodes(iso3)")
cur.execute("CREATE INDEX idx_nodes_year ON nodes(year)")
cur.execute("CREATE INDEX idx_edges_reporter ON edges(reporter_iso3)")
cur.execute("CREATE INDEX idx_edges_partner ON edges(partner_iso3)")
cur.execute("CREATE INDEX idx_edges_period ON edges(period)")
cur.execute("CREATE INDEX idx_edges_flow ON edges(flow_code)")
conn.commit()
conn.close()

# Schema document
schema_text = """================================================================
DATABASE SCHEMA: european_trade.db
================================================================

3 tables linked by iso3 as the common key.

TABLE 1: countries (50 rows)
  iso3          TEXT  PK   ISO3 country code (e.g. DEU)
  countryname   TEXT       Full country name
  capital       TEXT       Capital city
  lat           REAL       Latitude of capital
  lon           REAL       Longitude of capital
  comtradenum   INT  UQ   UN Comtrade numeric code
  Missing values: None

TABLE 2: nodes (205 rows)
  iso3            TEXT  FK->countries  ISO3 country code
  year            INT                  Year (2019-2023)
  gdpusd          REAL                 GDP in current USD
  population      REAL                 Population count
  tradepctgdp     REAL                 Trade as pct of GDP
  fdiinflowpctgdp REAL                 FDI inflow as pct of GDP
  Missing values: None within 205 rows
  Missing countries: AND, CHI, FRO, GIB, GRL, IMN, LIE, MCO, SMR

TABLE 3: edges (~15,000 rows)
  reporter_iso3   TEXT  FK->countries  Reporter country
  reporter_name   TEXT                 Reporter name
  partner_iso3    TEXT  FK->countries  Partner country
  partner_name    TEXT                 Partner name
  period          TEXT                 Year (2019-2023)
  flow_code       TEXT                 M=Import, X=Export
  total_trade_usd REAL                 Trade value in USD

RELATIONSHIPS:
  countries.iso3 <- nodes.iso3
  countries.iso3 <- edges.reporter_iso3
  countries.iso3 <- edges.partner_iso3

SOURCES:
  countries: World Bank API + REST Countries + Nominatim geocoding
  nodes:     World Bank API (GDP, population, trade/GDP, FDI/GDP)
  edges:     UN Comtrade API v1 (HS 2-digit annual bilateral trade)

PIPELINE:
  World Bank -> countries.csv, nodes.csv
  UN Comtrade API -> eu_comtrade.db (raw, ~2.8M rows)
  -> SQL aggregation -> bilateral_edges.db -> edges.csv
  -> countries.csv + nodes.csv + edges.csv -> european_trade.db
"""

with open(str(SCHEMA_PATH), "w") as f:
    f.write(schema_text)

print(schema_text)
print(f"Saved {DB_FINAL.name} and {SCHEMA_PATH.name}")

## Step 6. Validation & example queries

In [ ]:
conn = sqlite3.connect(str(DB_FINAL))

# Missing value analysis
print("=" * 60)
print("MISSING VALUE ANALYSIS")
print("=" * 60)

for table in ["countries", "nodes", "edges"]:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    print(f"\n--- {table} ({len(df)} rows) ---")
    has_issue = False
    for col in df.columns:
        null_n = df[col].isna().sum()
        zero_n = (
            (df[col] == 0).sum()
            if df[col].dtype in ["float64", "int64"]
            else 0
        )
        if null_n > 0 or zero_n > 0:
            print(f"  {col}: NULL={null_n}, ZERO={zero_n}")
            has_issue = True
    if not has_issue:
        print("  Complete, no missing values.")

# Countries missing from nodes
print("\n--- Countries missing from nodes ---")
missing = pd.read_sql("""
    SELECT c.iso3, c.countryname
    FROM countries c
    LEFT JOIN (SELECT DISTINCT iso3 FROM nodes) n
        ON c.iso3 = n.iso3
    WHERE n.iso3 IS NULL
""", conn)
for _, r in missing.iterrows():
    print(f"  {r['iso3']} ({r['countryname']})")

# Edge coverage
print("\n--- Edge coverage ---")
print(pd.read_sql("""
    SELECT period, flow_code, COUNT(*) AS rows,
           COUNT(DISTINCT reporter_iso3) AS reporters,
           COUNT(DISTINCT partner_iso3) AS partners
    FROM edges
    GROUP BY period, flow_code
    ORDER BY period, flow_code
""", conn).to_string(index=False))

# Example queries
print("\n" + "=" * 60)
print("EXAMPLE QUERIES")
print("=" * 60)

print("\nQ1: Germany top 5 export partners (2023)")
print(pd.read_sql("""
    SELECT partner_iso3, partner_name,
           ROUND(total_trade_usd / 1e9, 2) AS billion_usd
    FROM edges
    WHERE reporter_iso3 = 'DEU'
      AND period = '2023'
      AND flow_code = 'X'
    ORDER BY total_trade_usd DESC
    LIMIT 5
""", conn).to_string(index=False))

print("\nQ2: Total European trade by year and flow")
print(pd.read_sql("""
    SELECT period, flow_code,
           ROUND(SUM(total_trade_usd) / 1e9, 2) AS total_billion_usd
    FROM edges
    GROUP BY period, flow_code
    ORDER BY period, flow_code
""", conn).to_string(index=False))

print("\nQ3: Trade connectivity per country (2023)")
print(pd.read_sql("""
    SELECT reporter_iso3, reporter_name,
           COUNT(DISTINCT partner_iso3) AS num_partners
    FROM edges
    WHERE period = '2023'
    GROUP BY reporter_iso3
    ORDER BY num_partners DESC
    LIMIT 10
""", conn).to_string(index=False))

conn.close()

## Step 7. 2023 network visualization

In [ ]:
countries_plot = pd.read_csv(COUNTRIES_CSV)
nodes_plot = pd.read_csv(NODES_CSV)
edges_plot = pd.read_csv(EDGES_CSV)

# Merge nodes 2023 with coordinates
nodes_2023 = nodes_plot[nodes_plot["year"] == YEAR_FOR_VIS].merge(
    countries_plot[["iso3", "countryname", "lat", "lon"]],
    on="iso3",
    how="left",
)

# Top 30 bilateral trade pairs (M + X combined)
edges_2023 = (
    edges_plot[edges_plot["period"] == YEAR_FOR_VIS]
    .groupby(["reporter_iso3", "partner_iso3"], as_index=False)["total_trade_usd"]
    .sum()
    .nlargest(30, "total_trade_usd")
)

# Merge coordinates onto edges
coord = countries_plot[["iso3", "lon", "lat"]]
edges_2023 = edges_2023.merge(
    coord, left_on="reporter_iso3", right_on="iso3"
).rename(columns={"lon": "lon_s", "lat": "lat_s"}).drop(columns="iso3")
edges_2023 = edges_2023.merge(
    coord, left_on="partner_iso3", right_on="iso3"
).rename(columns={"lon": "lon_t", "lat": "lat_t"}).drop(columns="iso3")

# ----- Plot -----
fig, (ax, ax_leg) = plt.subplots(
    1, 2, figsize=(20, 12), gridspec_kw={"width_ratios": [3, 1]}
)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Draw edges
max_trade = edges_2023["total_trade_usd"].max()
for _, e in edges_2023.iterrows():
    w = 0.4 + (e["total_trade_usd"] / max_trade) * 4.5
    a = 0.15 + (e["total_trade_usd"] / max_trade) * 0.4
    ax.plot(
        [e.lon_s, e.lon_t], [e.lat_s, e.lat_t],
        color="#7f8c8d", alpha=a, linewidth=w,
        zorder=1, solid_capstyle="round",
    )

# Draw nodes
max_gdp = nodes_2023["gdpusd"].max()
sizes = 15 + (nodes_2023["gdpusd"] / max_gdp) * 800
scatter = ax.scatter(
    nodes_2023["lon"], nodes_2023["lat"],
    s=sizes,
    c=nodes_2023["gdpusd"] / max_gdp,
    cmap="Blues",
    edgecolors="white",
    linewidths=0.8,
    zorder=3,
    vmin=-0.3, vmax=1.0, alpha=0.85,
)

# Label top 12
text_fx = [pe.withStroke(linewidth=2.5, foreground="white")]
for _, row in nodes_2023.nlargest(12, "gdpusd").iterrows():
    ax.annotate(
        row["iso3"], (row["lon"], row["lat"]),
        xytext=(6, 6), textcoords="offset points",
        fontsize=9, fontweight="bold", color="#2c3e50",
        path_effects=text_fx, zorder=5,
    )

ax.set_xlim(-28, 42)
ax.set_ylim(33, 72)
ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)
ax.set_title(
    "European Economic Network 2023\n"
    "(GDP sized nodes | Bilateral trade links)",
    fontsize=14, fontweight="bold", color="#2c3e50", pad=12,
)
ax.grid(True, alpha=0.12)

# ----- Legend panel -----
ax_leg.set_facecolor("#1a2332")
ax_leg.set_xlim(0, 1)
ax_leg.set_ylim(0, 1)
ax_leg.axis("off")

ax_leg.text(
    0.5, 0.94, "How to Read This Chart",
    fontsize=15, fontweight="bold", color="white", ha="center",
)

items = [
    (0.82, "Node size", "Proportional to GDP — Germany,\nUK, France appear largest"),
    (0.67, "Node color", "Darker blue = larger economy\n(Blues colormap)"),
    (0.52, "Edge lines", "Top 30 bilateral trade links\nby total trade volume (2023)"),
    (0.37, "Edge width", "Thicker = higher trade value"),
    (0.22, "Labels", "Top 12 economies by GDP\nlabeled with ISO3 code"),
]
for y, title, desc in items:
    ax_leg.add_patch(plt.Rectangle(
        (0.06, y - 0.04), 0.03, 0.08,
        facecolor="#2ec4b6", transform=ax_leg.transAxes, clip_on=False,
    ))
    ax_leg.text(
        0.14, y + 0.02, title,
        fontsize=12, fontweight="bold", color="white",
        transform=ax_leg.transAxes,
    )
    ax_leg.text(
        0.14, y - 0.04, desc,
        fontsize=8.5, color="#aabbcc",
        transform=ax_leg.transAxes, linespacing=1.4,
    )

ax_leg.text(
    0.5, 0.07,
    "Data: UN Comtrade + World Bank\n50 European countries, 2023",
    fontsize=8.5, color="#2ec4b6", ha="center", style="italic",
    bbox=dict(
        boxstyle="round,pad=0.5", facecolor="#2a3a4a",
        edgecolor="#2ec4b6", linewidth=0.8,
    ),
    transform=ax_leg.transAxes,
)

plt.tight_layout()
plt.savefig(str(FIG_PATH), dpi=180, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved {FIG_PATH}")

## Download all outputs

In [ ]:
from google.colab import files

for f in [COUNTRIES_CSV, NODES_CSV, EDGES_CSV, DB_FINAL, SCHEMA_PATH, FIG_PATH]:
    if f.exists():
        files.download(str(f))
        print(f"Downloaded {f.name}")